# Nuvoton Industry Project -- Elevator People Counting on the M55M1

This notebook covers the following steps to calibrate and export the YOLO model onto monochrome device specifications:
1. **Conda environment setup** running inside Colab with `condacolab`.
2. **Dataset downloads and layouts** from HuggingFace to a 90/10 layout split.
3. **Local training running** with constant 192x192 monochrome variables triggers inputs.
4. **Evaluation metrics validation** layout scripts.
5. **Onnx/TFLite layout conversion exports** matching Ethos-U55 parameters.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [1]:
!conda --version

conda 25.3.1


In [2]:
%cd /content
!rm -rf m55m1-ElevatorCounting-YOLOv8n
GIT_REPO_URL = "https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n"
!git clone {GIT_REPO_URL}

import os
dir_name = os.path.basename(GIT_REPO_URL).replace('.git', '')
%cd {dir_name}/yolov8_ultralytics

/content
Cloning into 'm55m1-ElevatorCounting-YOLOv8n'...
remote: Enumerating objects: 974, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 974 (delta 16), reused 17 (delta 5), pack-reused 935 (from 2)
Receiving objects: 100% (974/974), 151.28 MiB | 34.49 MiB/s, done.
Resolving deltas: 100% (250/250), done.
/content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics


In [3]:
!cat /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/build/lib/ultralytics/cfg/models/v8/relu6-yolov8.yaml

# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLOv8 object detection model with P3-P5 outputs. For Usage examples see https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 1   # number of classes (overhead person detection)
scales: # model compound scaling constants, i.e. 'model=yolov8n.yaml' will call yolov8.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]  # YOLOv8n summary: 225 layers,  3157200 parameters,  3157184 gradients,   8.9 GFLOPs
  s: [0.33, 0.50, 1024]  # YOLOv8s summary: 225 layers, 11166560 parameters, 11166544 gradients,  28.8 GFLOPs
  m: [0.67, 0.75, 768]   # YOLOv8m summary: 295 layers, 25902640 parameters, 25902624 gradients,  79.3 GFLOPs
  l: [1.00, 1.00, 512]   # YOLOv8l summary: 365 layers, 43691520 parameters, 43691504 gradients, 165.7 GFLOPs
  x: [1.00, 1.25, 512]   # YOLOv8x summary: 365 layers, 68229648 parameters, 68229632 gradients, 258.5 GFLOPs

activation: nn.ReLU6()

# YOLOv8.0n backbone
backbone:
  # [from, repeats, module, 

In [4]:
!conda create --name yolov8_DG python=3.10 -y

!conda run -n yolov8_DG python -m pip install --upgrade pip setuptools
!conda run -n yolov8_DG python -m pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118

# Install current local package using .[export]
!conda run -n yolov8_DG python -m pip install .[export]

# Additional supporting packages needed for download/export
!conda run -n yolov8_DG python -m pip install datasets onnx2tf onnx
!conda run -n yolov8_DG python -m pip install matplotlib
!conda run -n yolov8_DG python -m pip install py-cpuinfo

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / done


==> WARNING: A newer version of conda exists. <==
    current version: 25.3.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/yolov8_DG

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    icu-78.3                   |       h33c6efd_0        12.1 MB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.7.4             |       hecca717_0          75 KB  conda-forge


In [5]:
# Download bdanko/overhead-person-detection dataset
# more info https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n
!conda run -n yolov8_DG python download_hf_dataset.py --dataset bdanko/overhead-person-detection --out-dir datasets/overhead_monochrome

Loading HuggingFace dataset: bdanko/overhead-person-detection
Total items: 13448
Populating local environment folder splits...
Processing split: train (12103 items)
Processing split: val (1345 items)

Done! Dataset structure created at: /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/datasets/overhead_monochrome
You can train with: --data datasets/overhead_monochrome/data.yaml


Generating train split: 100%|██████████| 13448/13448 [00:03<00:00, 4198.34 examples/s]

100%|██████████| 12103/12103 [00:46<00:00, 259.89it/s]

100%|██████████| 1345/1345 [00:05<00:00, 239.98it/s]



In [6]:
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_train.py \
  --model-cfg relu6-yolov8.yaml \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --weights yolov8n.pt \
  --epochs 10 \
  --device 0

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 319/451 items from pretrained weights
{'cfg': None, 'data': 'datasets/overhead_monochrome/data.yaml', 'epochs': 10, 'batch': 64, 'imgsz': 192, 'device': ['0'], 'optimizer': 'SGD', 'workers': 2, 'project': 'runs/train', 'name': 'exp', 'exist_ok': False, 'patience': 0, 'cache': False, 'close_mosaic': 3, 'resume': False, 'lr0': 0.01, 'lrf': 0.01, 'cos_lr': False, 'scale': 0.5, 'mixup': 0.0, 'copy_paste': 0.0}
New https://pypi.org/project/ultralytics/8.4.26 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=relu6-yolov8.yaml, data=datasets/overhead_monochrome/data.yaml, epochs=10, time=None, patience=0, batch=64, imgsz=192, save=True, save_period=-1, cache=False, device=['0'], workers=2, project=runs/train, name=exp, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, deter

In [9]:
# Confirm the model targets
# Note the experimental run you saved at!
EXPERIMENT="exp"

from ultralytics import YOLO
model = YOLO(f"runs/train/{EXPERIMENT}/weights/best.pt")
print(model.names)           # should show {0: 'person'} only
print(len(model.names))      # should be 1

{0: 'person'}
1


In [10]:
# validation
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_val.py \
  --weights runs/train/{EXPERIMENT}/weights/best.pt \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --device 0

{'weights': 'runs/train/exp/weights/best.pt', 'data': 'datasets/overhead_monochrome/data.yaml', 'annotations': None, 'device': '0', 'imgsz': 192, 'no_separate_outputs': False}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients
                   all       1345       2868      0.913      0.852      0.923      0.526
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
Speed: 0.0ms preprocess, 1.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/runs/detect/val


val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/datasets/overhead_monochrome/val/labels.cache... 1345 images, 55 backgrounds, 0 corrupt: 100%|██████████| 1345/1345 [00:00<?, ?it/s]
val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/data

In [11]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192

{'format': 'onnx', 'weights': './runs/train/exp/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.31'] not found, attempting AutoUpdate...


requirements: AutoUpdate success ✅ 1.2s, installed 1 package: ['onnxslim>=0.1.31']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 3.0s, saved as 'runs/train/exp/weights/best.onnx' (11.5 MB)

Export complete (4.4s)
Results saved to /content/m5

In [12]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192

{'format': 'onnx', 'weights': './runs/train/exp/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)

ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.1s, saved as 'runs/train/exp/weights/best.onnx' (11.5 MB)

Export complete (2.4s)
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/runs/train/exp/weights
Predict:         yolo predict task=detect model=runs/train/exp/weights/best.onnx imgsz=192  
Validate:        yolo val task=detect model=runs/train/exp/weights/best.onnx imgsz=192 data=da

In [13]:
# calibrations referencing saved monochrome folder
!conda run -n yolov8_DG env MPLBACKEND=Agg python generate_calib_data.py \
    --img-size 192 192 --n-img 100 -o calib_data_192_n100.npy \
    --img-dir datasets/overhead_monochrome/train/images

12103
convert from BGR to RGB format!!
(100, 192, 192, 3)
calib_datas.shape: (100, 192, 192, 3)



In [14]:
# Run onnx2tf to extract TFLite integers graph fully
!rm -rf saved_model
!conda run -n yolov8_DG env MPLBACKEND=Agg onnx2tf -i runs/train/{EXPERIMENT}/weights/best.onnx -oiqt -iqd int8 -oqd int8 -cind images calib_data_192_n4_rgb.npy "[[[[0,0,0]]]]" "[[[[1,1,1]]]]"


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 6              │ 6                │
│ Clip       │ 65             │ 65               │
│ Concat     │ 13             │ 13               │
│ Constant   │ 147            │ 147              │
│ Conv       │ 71             │ 71               │
│ MaxPool    │ 3              │ 3                │
│ Reshape    │ 6              │ 6                │
│ Resize     │ 2              │ 2                │
│ Transpose  │ 6              │ 6                │
│ Model Size │ 11.5MiB        │ 11.5MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━

In [15]:
# confirming int8 export

import tensorflow as tf

model_path = "saved_model/best_full_integer_quant.tflite"
interpreter = tf.lite.Interpreter(model_path=model_path)

details = interpreter.get_input_details()[0]
print(f"Name:  {details['name']}")
print(f"Shape: {details['shape']}")
print(f"DType: {details['dtype']}")

details = interpreter.get_output_details()[0]
print(f"Name:  {details['name']}")
print(f"Shape: {details['shape']}")
print(f"DType: {details['dtype']}")

Name:  serving_default_images:0
Shape: [  1 192 192   3]
DType: <class 'numpy.int8'>
Name:  PartitionedCall:4
Shape: [  1 144   1]
DType: <class 'numpy.int8'>


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [16]:
!conda run -n yolov8_DG python -m pip install ethos-u-vela

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.4 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 1.29.24 requires flatbuffers==25.12.19, but you have flatbuffers 24.3.25 which is incompatible.



In [17]:
!rm /content/best_integer_quant_vela.tflite
!conda run -n yolov8_DG vela saved_model/best_full_integer_quant.tflite \
    --accelerator-config ethos-u55-128 \
    --output-dir /content/ \
    --optimise Performance


rm: cannot remove '/content/best_integer_quant_vela.tflite': No such file or directory

Network summary for best_full_integer_quant
Accelerator configuration               Ethos_U55_128
System configuration             Ethos_U55_High_End_Embedded
Memory mode                               Shared_Sram
Accelerator clock                                 500 MHz
Design peak SRAM bandwidth                       3.73 GB/s
Design peak Off-chip Flash bandwidth             0.47 GB/s

Total SRAM used                                266.91 KiB
Total Off-chip Flash used                     2630.56 KiB

CPU operators = 0 (0.0%)
NPU operators = 156 (100.0%)

Average SRAM bandwidth                           1.89 GB/s
Input   SRAM bandwidth                           8.00 MB/batch
Weight  SRAM bandwidth                           9.18 MB/batch
Output  SRAM bandwidth                           3.77 MB/batch
Total   SRAM bandwidth                          21.18 MB/batch
Total   SRAM bandwidth            per i

In [18]:
!ls /content/*_vela.tflite
!ls runs/train/{EXPERIMENT}/weights

/content/best_full_integer_quant_vela.tflite
best.onnx  best.pt  last.pt


In [20]:
!cp /content/best_full_integer_quant_vela.tflite /content/YOLOv8n-od.tflite
!cp runs/train/{EXPERIMENT}/weights/best.onnx /content/YOLOv8n-od.onnx
!cp runs/train/{EXPERIMENT}/weights/best.pt /content/YOLOv8n-od.pt
